# Bilişsel Performans Skoru Tahmini - Pipeline v9 Enhanced
**V7 yapısı + 1.17536 submission özellikleri + Dataset analizi**

**V7'den farklı olan:**
1. ~30 yeni özellik (severity flags, latent indices, domain interactions)
2. Log transformasyonları (5 sütun)
3. Missing value flags (6 sütun)
4. OOF Target Encoding (leakage-free)
5. Group Statistics
6. XGBoost 3. model olarak eklendi
7. Meslek risk skoru + Ruh sağlığı severity mapping

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from catboost import CatBoostRegressor
import xgboost as xgb
import optuna
from scipy.optimize import minimize
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

TARGET = 'bilissel_performans_skoru'
ID_COL = 'id'
SEED = 42
SEEDS = [42, 123, 2024, 7, 99, 314, 555, 777, 1234, 2025]
N_SPLITS = 15
N_TRIALS = 80

print("Kütüphaneler yüklendi.")
print(f"Seeds: {len(SEEDS)} adet, Folds: {N_SPLITS}, Trials: {N_TRIALS}")

Kütüphaneler yüklendi.
Seeds: 10 adet, Folds: 15, Trials: 80


## 1. Veri Yükleme

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Eğitim: {train.shape}, Test: {test.shape}")
print(f"\nEksik Değerler:")
print(train.isnull().sum()[train.isnull().sum() > 0])

Eğitim: (56000, 24), Test: (24000, 23)

Eksik Değerler:
meslek                   1378
vucut_kitle_indeksi      1752
uyku_oncesi_kafein_mg    1463
stres_skoru              1715
kronotip                 1968
ruh_sagligi_durumu       1096
dtype: int64


## 2. Gelişmiş Özellik Mühendisliği (V9 — ~50+ özellik)

In [3]:
def feature_engineering(df):
    df = df.copy()
    eps = 1e-3

    # ── Ülke ve Meslek Normalizasyonu ──
    ulke_map = {'Spain': 'Ispanya', 'South Korea': 'Guney Kore', 'Sweden': 'Isvec',
                'Mexico': 'Meksika', 'Arjantin': 'Arjantin', 'Netherlands': 'Hollanda'}
    if 'ulke' in df.columns:
        df['ulke'] = df['ulke'].replace(ulke_map)
    if 'meslek' in df.columns:
        df['meslek'] = df['meslek'].replace({'Lawyer': 'Avukat'})

    # ── Missing Value Flags ──
    for col in ['kronotip', 'vucut_kitle_indeksi', 'stres_skoru',
                'uyku_oncesi_kafein_mg', 'meslek', 'ruh_sagligi_durumu']:
        if col in df.columns:
            df[f'{col}_eksik'] = df[col].isna().astype(int)

    # ── Uyku Kalitesi Temel ──
    df['toplam_kaliteli_uyku_yuzdesi'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['kalitesiz_uyku_yuzdesi'] = 100 - df['toplam_kaliteli_uyku_yuzdesi']
    df['rem_derin_orani'] = df['rem_yuzdesi'] / (df['derin_uyku_yuzdesi'] + eps)
    df['uyku_kalitesi_orani'] = df['derin_uyku_yuzdesi'] / (df['rem_yuzdesi'] + 1)

    # ── Uyku Bozulması ──
    df['uyku_bolunmesi_skoru'] = df['gecelik_uyanma_sayisi'] * df['uykuya_dalma_suresi_dk']
    df['uyku_bolunme_skoru_v2'] = df['uykuya_dalma_suresi_dk'] + df['gecelik_uyanma_sayisi'] * 12
    df['uyku_bozulma_genel'] = (
        df['gecelik_uyanma_sayisi'] +
        df['uykuya_dalma_suresi_dk'] / 10 +
        df['kalitesiz_uyku_yuzdesi'] / 10
    )

    # ── Stres Etkileri ──
    df['stres_x_calisma'] = df['stres_skoru'] * df['gunluk_calisma_saati']
    df['stres_x_uyanma'] = df['stres_skoru'] * df['gecelik_uyanma_sayisi']
    df['stres_x_kalitesiz_uyku'] = df['stres_skoru'] * df['kalitesiz_uyku_yuzdesi']

    # ── Ekran / Kafein / Uyarıcı ──
    df['ekran_kafein_toplam'] = df['uyku_oncesi_ekran_suresi_dk'] + df['uyku_oncesi_kafein_mg']
    df['negatif_etki_skoru'] = df['stres_skoru'] * df['ekran_kafein_toplam']
    df['uyarici_yuk'] = (df['uyku_oncesi_kafein_mg'].fillna(0) / 100
                         + df['uyku_oncesi_ekran_suresi_dk'] / 120)

    # ── Nabız / Aktivite / Sıcaklık ──
    df['nabiz_stres'] = df['dinlenik_nabiz_bpm'] * df['stres_skoru']
    df['aktivite_bin_adim'] = df['gunluk_adim_sayisi'] / 1000
    df['sicaklik_optimum_fark'] = (df['oda_sicakligi_celsius'] - 19.0).abs()
    df['hafta_sonu_abs_fark'] = df['hafta_sonu_uyku_farki_saat'].abs()

    # ── Log Transformasyonları ──
    for col in ['sekerleme_suresi_dk', 'uyku_oncesi_kafein_mg',
                'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi',
                'uykuya_dalma_suresi_dk']:
        df[f'{col}_log'] = np.log1p(df[col])

    # ── Ruh Sağlığı Severity Mapping ──
    mental_map = {
        'Saglikli': 0.0, 'Anksiyete': 1.0,
        'Depresyon': 2.0, 'Anksiyete ve depresyon': 3.0,
    }
    df['ruh_sagligi_severity'] = df['ruh_sagligi_durumu'].map(mental_map).fillna(1.0)

    # ── Meslek Risk Skoru ──
    job_risk_map = {
        'Saglik Personeli': 3.0, 'Avukat': 3.0,
        'Lojistik Calisani': 2.5, 'Ogrenci': 2.0, 'Yonetici': 2.0,
        'Satis ve Pazarlama Calisani': 1.5, 'Muhendis': 1.5,
        'Egitimci': 1.0, 'Serbest Calisan': 0.5, 'Ev Hanimi': 0.5,
        'Emekli': 0.0,
    }
    df['meslek_risk_skoru'] = df['meslek'].map(job_risk_map).fillna(1.5)

    # ── Severity / Threshold Flags ──
    df['high_stress_flag'] = (df['stres_skoru'] >= 8).astype(int)
    df['low_rem_flag'] = (df['rem_yuzdesi'] < 15).astype(int)
    df['low_deep_sleep_flag'] = (df['derin_uyku_yuzdesi'] < 15).astype(int)
    df['long_latency_flag'] = (df['uykuya_dalma_suresi_dk'] > 30).astype(int)
    df['many_wakeups_flag'] = (df['gecelik_uyanma_sayisi'] >= 6).astype(int)
    df['long_work_flag'] = (df['gunluk_calisma_saati'] > 9).astype(int)
    df['high_screen_flag'] = (df['uyku_oncesi_ekran_suresi_dk'] > 120).astype(int)
    df['high_caffeine_flag'] = (df['uyku_oncesi_kafein_mg'] > 100).astype(int)
    df['low_activity_flag'] = (df['gunluk_adim_sayisi'] < 4000).astype(int)
    df['extreme_temp_flag'] = (
        (df['oda_sicakligi_celsius'] < 17) | (df['oda_sicakligi_celsius'] > 23)
    ).astype(int)
    df['very_high_caffeine_flag'] = (df['uyku_oncesi_kafein_mg'] >= 200).astype(int)

    # ── Latent Composite Indices ──
    df['sleep_quality_index'] = (
        0.08 * df['rem_yuzdesi']
        + 0.10 * df['derin_uyku_yuzdesi']
        - 0.035 * df['uykuya_dalma_suresi_dk']
        - 0.22 * df['gecelik_uyanma_sayisi']
        - 0.004 * df['uyku_oncesi_ekran_suresi_dk']
        - 0.003 * df['uyku_oncesi_kafein_mg'].fillna(0)
        - 0.08 * df['sicaklik_optimum_fark']
    )
    df['fatigue_index'] = (
        0.65 * df['stres_skoru']
        + 0.30 * df['gunluk_calisma_saati']
        + 0.08 * df['dinlenik_nabiz_bpm']
        + 0.35 * df['gecelik_uyanma_sayisi']
        + 0.25 * df['ruh_sagligi_severity']
        + 0.18 * df['meslek_risk_skoru']
        - 0.00008 * df['gunluk_adim_sayisi']
    )
    df['recovery_index'] = (
        0.00025 * df['gunluk_adim_sayisi']
        + 0.015 * np.minimum(df['sekerleme_suresi_dk'], 45)
        - 0.010 * np.maximum(df['sekerleme_suresi_dk'] - 60, 0)
        + 0.25 * (df['gun_tipi'] == 'Hafta sonu').astype(int)
        - 0.20 * df['hafta_sonu_uyku_farki_saat'].abs()
    )
    df['net_recovery_balance'] = (
        df['sleep_quality_index'] + df['recovery_index'] - df['fatigue_index']
    )
    df['sleep_debt_proxy'] = (
        df['uyku_bolunme_skoru_v2']
        + df['gunluk_calisma_saati'] * 3
        + df['stres_skoru'] * 4
        - df['rem_yuzdesi']
        - df['derin_uyku_yuzdesi']
    )

    # ── Domain Interactions ──
    df['stress_x_mental_severity'] = df['stres_skoru'] * df['ruh_sagligi_severity']
    df['stress_x_job_risk'] = df['stres_skoru'] * df['meslek_risk_skoru']
    df['sleep_quality_x_stress'] = df['sleep_quality_index'] * df['stres_skoru']
    df['fatigue_x_weekday'] = (
        df['fatigue_index'] * (df['gun_tipi'] == 'Hafta ici').astype(int)
    )
    df['recovery_x_weekend'] = (
        df['recovery_index'] * (df['gun_tipi'] == 'Hafta sonu').astype(int)
    )
    df['gece_insani_hafta_ici'] = (
        (df['kronotip'] == 'Gece insani').astype(int)
        * (df['gun_tipi'] == 'Hafta ici').astype(int)
    )
    df['sabah_insani_hafta_sonu'] = (
        (df['kronotip'] == 'Sabah insani').astype(int)
        * (df['gun_tipi'] == 'Hafta sonu').astype(int)
    )
    df['bad_sleep_count'] = (
        df['low_rem_flag'] + df['low_deep_sleep_flag']
        + df['long_latency_flag'] + df['many_wakeups_flag']
        + df['high_screen_flag'] + df['high_caffeine_flag']
    )

    # ── Dataset Highlights: Kafein-Latency & Age-DeepSleep ──
    df['kafein_x_dalma_suresi'] = (
        df['uyku_oncesi_kafein_mg'].fillna(0) * df['uykuya_dalma_suresi_dk']
    )
    df['yas_x_derin_uyku'] = df['yas'] * df['derin_uyku_yuzdesi']

    # ── Kategorik Kombinasyonlar ──
    df['meslek_gun_tipi'] = df['meslek'].astype(str) + '_' + df['gun_tipi'].astype(str)
    df['ruh_sagligi_gun_tipi'] = df['ruh_sagligi_durumu'].astype(str) + '_' + df['gun_tipi'].astype(str)
    df['meslek_ruh_sagligi'] = df['meslek'].astype(str) + '_' + df['ruh_sagligi_durumu'].astype(str)
    df['kronotip_gun_tipi'] = df['kronotip'].astype(str) + '_' + df['gun_tipi'].astype(str)
    df['mevsim_kronotip'] = df['mevsim'].astype(str) + '_' + df['kronotip'].astype(str)
    df['cinsiyet_meslek'] = df['cinsiyet'].astype(str) + '_' + df['meslek'].astype(str)
    df['meslek_kronotip'] = df['meslek'].astype(str) + '_' + df['kronotip'].astype(str)

    # ── Kare Özellikler ──
    df['stres_kare'] = df['stres_skoru'] ** 2
    df['rem_kare'] = df['rem_yuzdesi'] ** 2

    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print(f"Yeni özellik sayısı: {train_fe.shape[1] - train.shape[1]} adet eklendi.")
print(f"Train: {train_fe.shape}, Test: {test_fe.shape}")

Yeni özellik sayısı: 65 adet eklendi.
Train: (56000, 89), Test: (24000, 88)


## 3. OOF Target Encoding + Group Statistics

In [4]:
def oof_target_encoding(train_df, test_df, y, te_cols, n_splits=15, seed=42, smoothing=50.0):
    """Leakage-free OOF Target Encoding"""
    prior = y.mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for col in te_cols:
        if col not in train_df.columns:
            continue
        train_df[f'{col}_te'] = np.nan
        for tr_idx, va_idx in kf.split(train_df):
            tmp = pd.DataFrame({'key': train_df.iloc[tr_idx][col], 'target': y[tr_idx]})
            stats = tmp.groupby('key', dropna=False)['target'].agg(['mean', 'count'])
            encoded = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
            train_df.iloc[va_idx, train_df.columns.get_loc(f'{col}_te')] = (
                train_df.iloc[va_idx][col].map(encoded).fillna(prior).values
            )
        # Test: full train'den hesapla
        tmp = pd.DataFrame({'key': train_df[col], 'target': y})
        stats = tmp.groupby('key', dropna=False)['target'].agg(['mean', 'count'])
        encoded = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
        test_df[f'{col}_te'] = test_df[col].map(encoded).fillna(prior)
    return train_df, test_df


def add_group_statistics(train_df, test_df, group_keys, num_cols):
    """Group-level mean/std/diff features"""
    for key in group_keys:
        if key not in train_df.columns:
            continue
        for col in num_cols:
            if col not in train_df.columns:
                continue
            grp_mean = train_df.groupby(key, dropna=False)[col].mean()
            grp_std = train_df.groupby(key, dropna=False)[col].std().fillna(0)
            g_mean = float(train_df[col].mean())
            g_std = float(train_df[col].std())
            for frame in [train_df, test_df]:
                frame[f'{key}_{col}_grp_mean'] = frame[key].map(grp_mean).fillna(g_mean)
                frame[f'{key}_{col}_grp_std'] = frame[key].map(grp_std).fillna(g_std)
                frame[f'{key}_{col}_grp_diff'] = frame[col] - frame[f'{key}_{col}_grp_mean']
    return train_df, test_df

## 4. Veri Ön İşleme

In [5]:
y = train_fe[TARGET].values
train_df = train_fe.drop(columns=[TARGET, ID_COL])
test_ids = test_fe[ID_COL].values
test_df = test_fe.drop(columns=[ID_COL])

# OOF Target Encoding
TE_COLS = ['meslek', 'ruh_sagligi_durumu', 'gun_tipi', 'kronotip',
           'meslek_gun_tipi', 'meslek_ruh_sagligi', 'kronotip_gun_tipi',
           'cinsiyet_meslek', 'meslek_kronotip']
train_df, test_df = oof_target_encoding(train_df, test_df, y, TE_COLS, n_splits=N_SPLITS, seed=SEED)
print(f"OOF Target Encoding uygulandı: {len(TE_COLS)} kolon")

# Group Statistics
GRP_KEYS = ['meslek', 'ruh_sagligi_durumu', 'gun_tipi', 'meslek_gun_tipi', 'meslek_ruh_sagligi']
GRP_NUMS = ['stres_skoru', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'gunluk_calisma_saati', 'gecelik_uyanma_sayisi']
train_df, test_df = add_group_statistics(train_df, test_df, GRP_KEYS, GRP_NUMS)
print(f"Group Statistics uygulandı: {len(GRP_KEYS)} key × {len(GRP_NUMS)} num")

n_train = train_df.shape[0]
all_data = pd.concat([train_df, test_df], ignore_index=True)
num_cols = all_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()

# CatBoost için ayrı kopya
all_cat = all_data.copy()
for c in num_cols:
    all_cat[c] = all_cat[c].fillna(all_cat[c].median())
for c in cat_cols:
    all_cat[c] = all_cat[c].fillna('Bilinmiyor')
X_train_cat = all_cat.iloc[:n_train].reset_index(drop=True)
X_test_cat = all_cat.iloc[n_train:].reset_index(drop=True)
cat_feat_idx = [X_train_cat.columns.get_loc(c) for c in cat_cols]

# LightGBM / XGBoost için Label Encoding
for c in num_cols:
    all_data[c] = all_data[c].fillna(all_data[c].median())
for c in cat_cols:
    all_data[c] = all_data[c].fillna('Bilinmiyor')
for c in cat_cols:
    le = LabelEncoder()
    all_data[c] = le.fit_transform(all_data[c])
    all_data[c] = all_data[c].astype('category')

X_train = all_data.iloc[:n_train].reset_index(drop=True)
X_test = all_data.iloc[n_train:].reset_index(drop=True)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(kf.split(np.arange(n_train)))

print(f"\nHazır: X_train={X_train.shape}, X_test={X_test.shape}")
print(f"Kategorik={len(cat_cols)} adet, Numerik={len(num_cols)} adet")
print(f"Toplam özellik: {X_train.shape[1]}")

OOF Target Encoding uygulandı: 9 kolon
Group Statistics uygulandı: 5 key × 5 num

Hazır: X_train=(56000, 171), X_test=(24000, 171)
Kategorik=14 adet, Numerik=157 adet
Toplam özellik: 171


## 5. LightGBM - Optuna (80 Trial) + Multi-Seed (10)

In [6]:
def objective_lgb(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'random_state': 42, 'verbose': -1
    }
    X_tr, X_va, y_tr, y_va = train_test_split(X_train, y, test_size=0.2, random_state=42)
    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain, categorical_feature=cat_cols)
    m = lgb.train(params, dtrain, num_boost_round=1500, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    return root_mean_squared_error(y_va, m.predict(X_va, num_iteration=m.best_iteration))

print("LightGBM Optuna Optimizasyonu Başlıyor (80 trial)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"\nLightGBM Optuna En iyi RMSE: {study_lgb.best_value:.4f}")

# Multi-seed LightGBM
lgb_oof_all = np.zeros((len(SEEDS), n_train))
lgb_test_all = np.zeros((len(SEEDS), len(X_test)))
best_lgb_params = study_lgb.best_params
best_lgb_params.update({'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1})

for s_idx, seed in enumerate(SEEDS):
    params = best_lgb_params.copy()
    params['random_state'] = seed
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    folds_s = list(kf_s.split(np.arange(n_train)))
    oof = np.zeros(n_train)
    test_preds = np.zeros(len(X_test))
    for fold, (tr_idx, va_idx) in enumerate(folds_s):
        dtrain = lgb.Dataset(X_train.iloc[tr_idx], label=y[tr_idx], categorical_feature=cat_cols)
        dval = lgb.Dataset(X_train.iloc[va_idx], label=y[va_idx], reference=dtrain, categorical_feature=cat_cols)
        m = lgb.train(params, dtrain, num_boost_round=3000, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] = m.predict(X_train.iloc[va_idx], num_iteration=m.best_iteration)
        test_preds += m.predict(X_test, num_iteration=m.best_iteration) / len(folds_s)
    lgb_oof_all[s_idx] = oof
    lgb_test_all[s_idx] = test_preds
    print(f"LightGBM Seed {seed} OOF RMSE: {root_mean_squared_error(y, oof):.4f}")

lgb_oof = lgb_oof_all.mean(axis=0)
lgb_test = lgb_test_all.mean(axis=0)
print(f"\nLightGBM Multi-Seed OOF RMSE: {root_mean_squared_error(y, lgb_oof):.4f}")

LightGBM Optuna Optimizasyonu Başlıyor (80 trial)...


  0%|          | 0/80 [00:00<?, ?it/s]


LightGBM Optuna En iyi RMSE: 1.2243
LightGBM Seed 42 OOF RMSE: 1.2206
LightGBM Seed 123 OOF RMSE: 1.2203
LightGBM Seed 2024 OOF RMSE: 1.2208
LightGBM Seed 7 OOF RMSE: 1.2203
LightGBM Seed 99 OOF RMSE: 1.2201
LightGBM Seed 314 OOF RMSE: 1.2206
LightGBM Seed 555 OOF RMSE: 1.2201
LightGBM Seed 777 OOF RMSE: 1.2201
LightGBM Seed 1234 OOF RMSE: 1.2205
LightGBM Seed 2025 OOF RMSE: 1.2206

LightGBM Multi-Seed OOF RMSE: 1.2189


## 6. CatBoost - Optuna (80 Trial) + Multi-Seed (10)

In [7]:
def objective_cat(trial):
    params = {
        'iterations': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42, 'verbose': 0, 'early_stopping_rounds': 150,
        'cat_features': cat_feat_idx
    }
    X_tr, X_va, y_tr, y_va = train_test_split(X_train_cat, y, test_size=0.2, random_state=42)
    m = CatBoostRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    return root_mean_squared_error(y_va, m.predict(X_va))

print("CatBoost Optuna Optimizasyonu Başlıyor (80 trial)...")
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)
print(f"\nCatBoost Optuna En iyi RMSE: {study_cat.best_value:.4f}")

cat_oof_all = np.zeros((len(SEEDS), n_train))
cat_test_all = np.zeros((len(SEEDS), len(X_test)))
best_cat_params = study_cat.best_params

for s_idx, seed in enumerate(SEEDS):
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    folds_s = list(kf_s.split(np.arange(n_train)))
    oof = np.zeros(n_train)
    test_preds = np.zeros(len(X_test))
    for fold, (tr_idx, va_idx) in enumerate(folds_s):
        m_cat = CatBoostRegressor(
            iterations=2000,
            learning_rate=best_cat_params['learning_rate'],
            depth=best_cat_params['depth'],
            l2_leaf_reg=best_cat_params['l2_leaf_reg'],
            random_strength=best_cat_params['random_strength'],
            bagging_temperature=best_cat_params['bagging_temperature'],
            border_count=best_cat_params['border_count'],
            random_seed=seed, verbose=0, early_stopping_rounds=150,
            cat_features=cat_feat_idx
        )
        m_cat.fit(X_train_cat.iloc[tr_idx], y[tr_idx],
                  eval_set=(X_train_cat.iloc[va_idx], y[va_idx]), verbose=0)
        oof[va_idx] = m_cat.predict(X_train_cat.iloc[va_idx])
        test_preds += m_cat.predict(X_test_cat) / len(folds_s)
    cat_oof_all[s_idx] = oof
    cat_test_all[s_idx] = test_preds
    print(f"CatBoost Seed {seed} OOF RMSE: {root_mean_squared_error(y, oof):.4f}")

cat_oof = cat_oof_all.mean(axis=0)
cat_test = cat_test_all.mean(axis=0)
print(f"\nCatBoost Multi-Seed OOF RMSE: {root_mean_squared_error(y, cat_oof):.4f}")

CatBoost Optuna Optimizasyonu Başlıyor (80 trial)...


  0%|          | 0/80 [00:00<?, ?it/s]


CatBoost Optuna En iyi RMSE: 1.2183
CatBoost Seed 42 OOF RMSE: 1.2153
CatBoost Seed 123 OOF RMSE: 1.2154
CatBoost Seed 2024 OOF RMSE: 1.2153
CatBoost Seed 7 OOF RMSE: 1.2151
CatBoost Seed 99 OOF RMSE: 1.2151
CatBoost Seed 314 OOF RMSE: 1.2147
CatBoost Seed 555 OOF RMSE: 1.2151
CatBoost Seed 777 OOF RMSE: 1.2156
CatBoost Seed 1234 OOF RMSE: 1.2153
CatBoost Seed 2025 OOF RMSE: 1.2150

CatBoost Multi-Seed OOF RMSE: 1.2138


## 7. XGBoost - Optuna (80 Trial) + Multi-Seed (10)

In [8]:
def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse',
        'booster': 'gbtree', 'tree_method': 'hist', 'enable_categorical': True,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'random_state': 42, 'verbosity': 0,
    }
    X_tr, X_va, y_tr, y_va = train_test_split(X_train, y, test_size=0.2, random_state=42)
    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    m = xgb.train(params, dtrain, num_boost_round=1500,
                  evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    preds = m.predict(dval, iteration_range=(0, m.best_iteration + 1))
    return root_mean_squared_error(y_va, preds)

print("XGBoost Optuna Optimizasyonu Başlıyor (80 trial)...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"\nXGBoost Optuna En iyi RMSE: {study_xgb.best_value:.4f}")

xgb_oof_all = np.zeros((len(SEEDS), n_train))
xgb_test_all = np.zeros((len(SEEDS), len(X_test)))
best_xgb_params = study_xgb.best_params
best_xgb_params.update({
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'booster': 'gbtree', 'tree_method': 'hist', 'enable_categorical': True, 'verbosity': 0
})

for s_idx, seed in enumerate(SEEDS):
    params = best_xgb_params.copy()
    params['random_state'] = seed
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    folds_s = list(kf_s.split(np.arange(n_train)))
    oof = np.zeros(n_train)
    test_preds = np.zeros(len(X_test))
    dtest = xgb.DMatrix(X_test, enable_categorical=True)
    for fold, (tr_idx, va_idx) in enumerate(folds_s):
        dtrain = xgb.DMatrix(X_train.iloc[tr_idx], label=y[tr_idx], enable_categorical=True)
        dval = xgb.DMatrix(X_train.iloc[va_idx], label=y[va_idx], enable_categorical=True)
        m = xgb.train(params, dtrain, num_boost_round=3000,
                      evals=[(dval, 'val')], early_stopping_rounds=150, verbose_eval=False)
        oof[va_idx] = m.predict(dval, iteration_range=(0, m.best_iteration + 1))
        test_preds += m.predict(dtest, iteration_range=(0, m.best_iteration + 1)) / len(folds_s)
    xgb_oof_all[s_idx] = oof
    xgb_test_all[s_idx] = test_preds
    print(f"XGBoost Seed {seed} OOF RMSE: {root_mean_squared_error(y, oof):.4f}")

xgb_oof = xgb_oof_all.mean(axis=0)
xgb_test = xgb_test_all.mean(axis=0)
print(f"\nXGBoost Multi-Seed OOF RMSE: {root_mean_squared_error(y, xgb_oof):.4f}")

XGBoost Optuna Optimizasyonu Başlıyor (80 trial)...


  0%|          | 0/80 [00:00<?, ?it/s]


XGBoost Optuna En iyi RMSE: 1.2235
XGBoost Seed 42 OOF RMSE: 1.2196
XGBoost Seed 123 OOF RMSE: 1.2191
XGBoost Seed 2024 OOF RMSE: 1.2192
XGBoost Seed 7 OOF RMSE: 1.2189
XGBoost Seed 99 OOF RMSE: 1.2191
XGBoost Seed 314 OOF RMSE: 1.2187
XGBoost Seed 555 OOF RMSE: 1.2191
XGBoost Seed 777 OOF RMSE: 1.2193
XGBoost Seed 1234 OOF RMSE: 1.2198
XGBoost Seed 2025 OOF RMSE: 1.2191

XGBoost Multi-Seed OOF RMSE: 1.2177


## 8. Stacking Ensemble - RidgeCV + Scipy Minimize (3 Model)

In [9]:
X_meta_train = np.column_stack([lgb_oof, cat_oof, xgb_oof])
X_meta_test = np.column_stack([lgb_test, cat_test, xgb_test])

# Scipy Minimize
def rmse_objective(weights):
    pred = np.dot(X_meta_train, weights)
    return root_mean_squared_error(y, pred)

n_models = X_meta_train.shape[1]
w0 = [1.0 / n_models] * n_models
cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1)] * n_models
res = minimize(rmse_objective, w0, method='SLSQP', bounds=bounds, constraints=cons)
best_weights = res.x

print(f"--- Scipy Minimize Stacking ---")
print(f"Ağırlıklar: LGB={best_weights[0]:.4f}, CAT={best_weights[1]:.4f}, XGB={best_weights[2]:.4f}")
ens_oof_scipy = np.dot(X_meta_train, best_weights)
ens_test_scipy = np.dot(X_meta_test, best_weights)
print(f"Scipy Stacking OOF RMSE: {root_mean_squared_error(y, ens_oof_scipy):.5f}")

# RidgeCV Meta-Model
ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0])
ridge.fit(X_meta_train, y)
ens_oof_ridge = ridge.predict(X_meta_train)
ens_test_ridge = ridge.predict(X_meta_test)
print(f"\n--- RidgeCV Stacking ---")
print(f"RidgeCV alpha: {ridge.alpha_}")
print(f"RidgeCV Stacking OOF RMSE: {root_mean_squared_error(y, ens_oof_ridge):.5f}")

# En iyi ensemble seç
rmse_scipy = root_mean_squared_error(y, ens_oof_scipy)
rmse_ridge = root_mean_squared_error(y, ens_oof_ridge)

if rmse_scipy <= rmse_ridge:
    ens_oof = ens_oof_scipy
    ens_test = ens_test_scipy
    ens_method = "Scipy"
else:
    ens_oof = ens_oof_ridge
    ens_test = ens_test_ridge
    ens_method = "RidgeCV"

print(f"\nSeçilen Ensemble Yöntemi: {ens_method}")

--- Scipy Minimize Stacking ---
Ağırlıklar: LGB=0.0181, CAT=0.8667, XGB=0.1152
Scipy Stacking OOF RMSE: 1.21376

--- RidgeCV Stacking ---
RidgeCV alpha: 10.0
RidgeCV Stacking OOF RMSE: 1.21372

Seçilen Ensemble Yöntemi: RidgeCV


## 9. Sonuçlar ve Submission

In [10]:
results = pd.DataFrame({
    'Model': ['LightGBM (Multi-Seed)', 'CatBoost (Multi-Seed)',
              'XGBoost (Multi-Seed)', f'Ensemble ({ens_method})'],
    'OOF RMSE': [
        root_mean_squared_error(y, lgb_oof),
        root_mean_squared_error(y, cat_oof),
        root_mean_squared_error(y, xgb_oof),
        root_mean_squared_error(y, ens_oof)
    ],
    'OOF R2': [r2_score(y, lgb_oof), r2_score(y, cat_oof),
               r2_score(y, xgb_oof), r2_score(y, ens_oof)]
}).sort_values('OOF RMSE')
print("\n" + "=" * 60)
print("SONUÇLAR:")
print("=" * 60)
print(results.to_string(index=False))

# Akıllı clipping
final = np.clip(ens_test, y.min(), y.max())
sub = pd.DataFrame({'id': test_ids, 'bilissel_performans_skoru': final})
sub.to_csv('submission_v9_enhanced.csv', index=False)
print(f"\nsubmission_v9_enhanced.csv oluşturuldu! ({len(sub)} satır)")
print(sub.head())
print(f"\nTahmin istatistikleri:")
print(f"  Min: {final.min():.4f}, Max: {final.max():.4f}, Mean: {final.mean():.4f}, Std: {final.std():.4f}")


SONUÇLAR:
                Model  OOF RMSE   OOF R2
   Ensemble (RidgeCV)  1.213719 0.704233
CatBoost (Multi-Seed)  1.213810 0.704189
 XGBoost (Multi-Seed)  1.217653 0.702313
LightGBM (Multi-Seed)  1.218869 0.701718

submission_v9_enhanced.csv oluşturuldu! (24000 satır)
   id  bilissel_performans_skoru
0   1                   5.935625
1   2                   6.717801
2   3                   3.008045
3   4                   7.276788
4   5                   3.675449

Tahmin istatistikleri:
  Min: 0.0000, Max: 10.0000, Mean: 5.9417, Std: 1.8697
